In [9]:
# Add the path to the src module
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO

import sys
from pathlib import Path
import shutil

sys.path.append(
    '/data/c/yangyusheng_optimized/AFdb_based_domain_extraction/src/util')
from extract_pLDDT_functions import extract_protein_seq_pdb_gz
# load basic python modules

# 1.Load AF vs pombase data

In [10]:
pombase_vs_AFdb = pd.read_excel(
    "../results/coding_genes_AF_vs_pombase_summary.xlsx", sheet_name="coding_genes_AF_vs_pombase")

# 2.Load DY47073 (ref pombase 20240101) data

In [11]:
pseudogenic_transcript = ['SPAC212.05c.1', 'SPAC212.07c.1', 'SPAC212.09c.1', 'SPAC212.10.1', 'SPAC23D3.05c.1', 'SPAC977.13c.1', 'SPAPB24D3.05c.1', 'SPBC1348.11.1', 'SPBC1706.02c.1', 'SPBC18E5.15.1', 'SPBC31A8.02.1', 'SPBCPT2R1.05c.1', 'SPBCPT2R1.06c.1', 'SPBCPT2R1.07c.1', 'SPBCPT2R1.10.1',
                          'SPBPB10D8.03.1', 'SPBPB21E7.06.1', 'SPBPB21E7.08.1', 'SPCC132.05c.1', 'SPCC1450.01c.1', 'SPCC188.10c.1', 'SPCC18B5.02c.1', 'SPCC285.06c.1', 'SPCC306.10.1', 'SPCC548.02c.1', 'SPCC553.05c.1', 'SPCC576.16c.1', 'SPCC622.21.1', 'SPCC663.07c.1', 'SPCC830.02.1', 'SPCP20C8.03.1']

unannotated_pseudogene = ['SPAC750.08c.1', 'SPAC2E12.05.1', 'SPBC337.02c.1',
                          'SPAC29A4.23.1', 'SPBC530.06c.1', 'SPBPB21E7.02c.1', 'SPCC1494.11c.1', 'SPAC23A1.20.1']

In [12]:
peptide_file = Path(
    "../references/DY47073_v240116_combined_genome_liftoff_addBraker_240322_protein.fa")

# Parse the FASTA file and convert to DataFrame
records = list(SeqIO.parse(peptide_file, "fasta"))

transcript_pep = [
    (record.id.split(" ")[0],
     str(record.seq)) for record in records
]

remove_transcript = pseudogenic_transcript + unannotated_pseudogene
data = [pep for pep in transcript_pep if pep[0]
        not in remove_transcript]


def rename_id(pair):
    id = pair[0]
    pep = pair[1]

    if "_" in id:
        id_elements = id.split("_")
        if "braker" not in id:
            gene = id.split("_")[0]
            gene = re.search("(\S+)\.\d{1}", gene).group(1)
            copy = id.split("_")[1]
            rename_gene = gene + "_" + copy

            return (rename_gene, id, pep)
        elif ("braker" in id) & (len(id_elements) == 2):
            gene = id.split("_")[0] + "_" + id.split("_")[1]
            gene = re.search("(\S+)\.\d{1}", gene).group(1)
            rename_gene = gene
            return (rename_gene, id, pep)
        elif ("braker" in id) & (len(id_elements) == 3):
            gene = id.split("_")[0] + "_" + id.split("_")[1]
            gene = re.search("(\S+)\.\d{1}", gene).group(1)
            copy = id.split("_")[2]
            rename_gene = gene + "_" + copy
            return (rename_gene, id, pep)
    else:
        gene = re.search("(\S+)\.\d{1}", id).group(1)
        return (gene, id, pep)


data = [rename_id(pair) for pair in data]
DY47073_protein_stats = pd.DataFrame(
    data, columns=["gene_systematic_id", "transcript_id", "DY47073_seq"])
DY47073_protein_stats["DY47073_seq"] = DY47073_protein_stats["DY47073_seq"].str.rstrip(
    "*")
DY47073_protein_stats["DY47073_length"] = DY47073_protein_stats["DY47073_seq"].apply(
    len)

# 3.Merge data

In [13]:
coding_genes = pombase_vs_AFdb.merge(
    DY47073_protein_stats, on="gene_systematic_id", how="outer")

# 4.Compare DY47073 with pombase

In [14]:
def assign_type(row):

    Pombase_seq = row["Pombase_seq"]
    Pombase_length = row["Pombase_length"]
    AF_seq = row["AF_seq"]
    AF_length = row["AF_length"]
    DY47073_seq = row["DY47073_seq"]
    DY47073_length = row["DY47073_length"]

    if pd.notna(Pombase_seq) and pd.isna(DY47073_seq):
        return "no_this_protein_in_DY47073"
    elif pd.isna(Pombase_seq) and pd.notna(DY47073_seq):
        if pd.notna(AF_seq) and (AF_seq == DY47073_seq):
            return "no_this_protein_in_pombase_but_in_AFdb_the_same"
        elif pd.notna(AF_seq) and (AF_seq != DY47073_seq):
            return "no_this_protein_in_pombase_but_in_AFdb_different"
        elif pd.isna(AF_seq):
            return "no_this_protein_in_pombase_or_AFdb"

    elif pd.isna(Pombase_seq) and pd.isna(DY47073_seq):
        return "no_proteins"
    elif Pombase_seq == DY47073_seq:
        return "same_seq"
    elif (Pombase_seq != DY47073_seq):
        if (DY47073_seq == AF_seq):
            return "different_seq_with_pombase_but_same_with_AFdb"
        else:
            if (Pombase_length == DY47073_length):
                return "different_seq_same_length_with_pombase"
            elif (Pombase_length != DY47073_length):
                return "different_seq_different_length_with_pombase"
    else:
        return "undefined"


coding_genes["DY47073_type"] = coding_genes.apply(assign_type, axis=1)
print(coding_genes["DY47073_type"].value_counts())

DY47073_type
same_seq                                         5066
different_seq_same_length_with_pombase             36
no_proteins                                        26
different_seq_different_length_with_pombase        23
no_this_protein_in_pombase_or_AFdb                 16
no_this_protein_in_DY47073                         10
different_seq_with_pombase_but_same_with_AFdb       9
Name: count, dtype: int64


In [15]:
def assign_DY47073_source(row):
    if row["DY47073_type"] == "same_seq":
        return row["Source"]
    elif row["DY47073_type"] == "different_seq_with_pombase_but_same_with_AFdb":
        return "AFdb"
    elif row["DY47073_type"] in ["no_this_protein_in_pombase_or_AFdb", "different_seq_same_length_with_pombase", "different_seq_different_length_with_pombase"]:
        return "vDulab2"
    else:
        return "Not sure"


coding_genes["Source"] = coding_genes.apply(
    assign_DY47073_source, axis=1)

# Fill NA values in gene_name with gene_systematic_id
coding_genes['gene_name'] = coding_genes['gene_name'].fillna(coding_genes['gene_systematic_id'])

# 5.re-predict alphafold results

In [16]:
predicted_results_folder = Path(
    "../data/0_raw/20240724_predicted_results_renamed")
DY47073_predicted_results_folder = Path(
    "../data/0_raw/20240904_formated_results_vDulab2_noNaN")
AF_pdb_dir = Path(
    "/data/c/yangyusheng_optimized/resource/AlphaFold_Dataset/20230708_downloaded/UP000002485_284812_SCHPO_v4")

AFdb_vDulab_vDulab2_results_merged_folder = Path(
    "../data/1_AFdb_vDulab_vDulab2_results_merged")
AFdb_vDulab_vDulab2_results_merged_folder.mkdir(exist_ok=True)
(AFdb_vDulab_vDulab2_results_merged_folder / "PDB").mkdir(exist_ok=True)
(AFdb_vDulab_vDulab2_results_merged_folder / "PAE_files").mkdir(exist_ok=True)


AFdb_PDB_folder = AF_pdb_dir
AFdb_PAE_folder = AF_pdb_dir.parent / "PAE_files"
vDulab_PDB_folder = predicted_results_folder / "vDulab"
vDulab_PAE_folder = predicted_results_folder / "PAE_files"
vDulab2_PDB_folder = DY47073_predicted_results_folder / "vDulab2"
vDulab2_PAE_folder = DY47073_predicted_results_folder / "PAE_files"

AFdb_PDB_files = {
    file.name.split("-F1-")[0].lstrip("AF-"): file for file in AFdb_PDB_folder.glob("*.pdb.gz")
}
AFdb_PAE_files = {
    file.name.split("-F1-")[0].lstrip("AF-"): file for file in AFdb_PAE_folder.glob("*.json")
}
vDulab_PDB_files = {
    file.name.split("-F1-")[0].lstrip("AF-"): file for file in vDulab_PDB_folder.glob("*.pdb.gz")
}
vDulab_PAE_files = {
    file.name.split("-F1-")[0].lstrip("AF-"): file for file in vDulab_PAE_folder.glob("*.json")
}
vDulab2_PDB_files = {
    file.name.split("-F1-")[0].lstrip("AF-"): file for file in vDulab2_PDB_folder.glob("*.pdb.gz")
}
vDulab2_PAE_files = {
    file.name.split("-F1-")[0].lstrip("AF-"): file for file in vDulab2_PAE_folder.glob("*.json")
}

for idx, row in coding_genes[coding_genes["DY47073_type"].notna()].iterrows():
    uniprot_id = row["uniprot_id"]
    gene_systematic_id = row["gene_systematic_id"]
    gene_name = row["gene_name"]
    pombase_type = row["type"]
    seq_type = row["DY47073_type"]
    source = row["Source"]

    if source == "AFdb":
        pdb_file = AFdb_PDB_files.get(uniprot_id)
        pae_file = AFdb_PAE_files.get(uniprot_id)
        # coding_genes.loc[idx, "uniprot_id"] = uniprot_id
    elif source == "vDulab":
        if uniprot_id in vDulab_PDB_files:
            pdb_file = vDulab_PDB_files.get(uniprot_id)
            pae_file = vDulab_PAE_files.get(uniprot_id)
            coding_genes.loc[idx, "uniprot_id"] = uniprot_id
        elif gene_name in vDulab_PDB_files:
            pdb_file = vDulab_PDB_files.get(gene_name)
            pae_file = vDulab_PAE_files.get(gene_name)
            coding_genes.loc[idx, "uniprot_id"] = gene_name
        else:
            print(gene_systematic_id, gene_name, uniprot_id, pombase_type, seq_type, source)
    elif source == "vDulab2":
        if uniprot_id in vDulab2_PDB_files:
            pdb_file = vDulab2_PDB_files.get(uniprot_id)
            pae_file = vDulab2_PAE_files.get(uniprot_id)
            coding_genes.loc[idx, "uniprot_id"] = uniprot_id
        elif gene_name in vDulab2_PDB_files:
            pdb_file = vDulab2_PDB_files.get(gene_name)
            pae_file = vDulab2_PAE_files.get(gene_name)
            coding_genes.loc[idx, "uniprot_id"] = gene_name
        else:
            print(gene_systematic_id, gene_name, uniprot_id, pombase_type, seq_type, source)

    if pdb_file:
        shutil.copy(pdb_file, AFdb_vDulab_vDulab2_results_merged_folder / "PDB")
    if pae_file:
        shutil.copy(pae_file, AFdb_vDulab_vDulab2_results_merged_folder / "PAE_files")


SPAC1093.06c dhc1 O13290 no_prediction_proteins same_seq vDulab
SPAC1F5.11c tra2 Q10064 no_prediction_proteins same_seq vDulab
SPAC23G3.02c sib1 Q9P7T1 no_prediction_proteins same_seq vDulab
SPCC23B6.03c tel1 O74630 no_prediction_proteins different_seq_different_length_with_pombase vDulab2
SPCC737.08 mdn1 O94248 no_prediction_proteins same_seq vDulab
SPBCPT2R1.06c_1 SPBCPT2R1.06c_1 nan nan no_this_protein_in_pombase_or_AFdb vDulab2
SPBCPT2R1.07c_1 SPBCPT2R1.07c_1 nan nan no_this_protein_in_pombase_or_AFdb vDulab2


# 6.Output the fasta file for the inconsistent proteins

In [17]:
def write_fasta(df, outputFolder):
    outputFolder.mkdir(parents=True, exist_ok=True)
    for index, row in df.iterrows():
        with open(outputFolder / (row["gene_systematic_id"] + ".fasta"), "w") as f:
            f.write(">" + row["gene_systematic_id"] + "\n")
            f.write(row["DY47073_seq"] + "\n")

In [18]:
fasta_output_folder = Path("../results/DY47073_fasta_for_customed_prediction")
fasta_output_folder.mkdir(exist_ok=True)

no_this_protein_in_pombase_or_AFdb = coding_genes.query(
    "DY47073_type == 'no_this_protein_in_pombase_or_AFdb'")
different_seq_same_length_with_pombase = coding_genes.query(
    "DY47073_type == 'different_seq_same_length_with_pombase'")
different_seq_different_length_with_pombase = coding_genes.query(
    "DY47073_type == 'different_seq_different_length_with_pombase'")

with pd.ExcelWriter("../results/coding_genes_AF_vs_pombase_vs_DY47073_summary_20240912_updated.xlsx") as xlsx:
    coding_genes.to_excel(
        xlsx, sheet_name="coding_genes_AF_vs_pombe_vs_DY", index=False, header=True)
    no_this_protein_in_pombase_or_AFdb.to_excel(
        xlsx, sheet_name="no_protein_in_pombe_or_AFdb", index=False, header=True)
    different_seq_same_length_with_pombase.to_excel(
        xlsx, sheet_name="diff_seq_same_len_with_pombe", index=False, header=True)
    different_seq_different_length_with_pombase.to_excel(
        xlsx, sheet_name="diff_seq_diff_len_with_pombe", index=False, header=True)


# write_fasta(no_this_protein_in_pombase_or_AFdb, fasta_output_folder /
#             "no_this_protein_in_pombase_or_AFdb")
# write_fasta(different_seq_same_length_with_pombase, fasta_output_folder /
#             "different_seq_same_length_with_pombase")
# write_fasta(different_seq_different_length_with_pombase, fasta_output_folder /
#             "different_seq_different_length_with_pombase")

In [16]:
coding_genes[coding_genes.duplicated(["gene_systematic_id", "DY47073_seq"], keep=False)].to_excel(
    "../results/duplicated_proteins_in_DY47073.xlsx", index=False, header=True)

In [ ]:

# # pLDDT_score = pd.read_csv(
# #     "../results/UP000002485_284812_SCHPO_v4_pLDDT_scores.tsv", sep="\t")
# # pLDDT_score["mean_pLDDT"] = pLDDT_score["pLDDT"].str.split(
# #     ",", expand=True).astype(float).mean(axis=1).round(2)


# def count_differences(str1, str2):
#     return sum(c1 != c2 for c1, c2 in zip(str1, str2))


# coding_genes.loc[(coding_genes["same_length_different_seq"] == "YES"), "same_length_different_seq_aa_count"] = coding_genes.loc[(
#     coding_genes["same_length_different_seq"] == "YES"), :].apply(lambda row: count_differences(row["Pombase_seq"], row["AF_seq"]), axis=1)

# coding_genes["length_difference"] = coding_genes["Pombase_length"] - \
#     coding_genes["AF_length"]

# coding_genes["Pombase_seq_start_codon"] = coding_genes["Pombase_seq"].str[0]
# coding_genes["AF_seq_start_codon"] = coding_genes["AF_seq"].str[0]

# coding_genes = coding_genes.merge(
#     pLDDT_score[["uniprot_id", "mean_pLDDT"]], on="uniprot_id", how="left")

# Assign the source for the annotated PAE domain

In [19]:
coding_genes = pd.read_excel(
    "../results/coding_genes_AF_vs_pombase_vs_DY47073_summary_20240912_updated.xlsx", sheet_name="coding_genes_AF_vs_pombe_vs_DY")
annotated_PAE_domain = pd.read_csv(
    "../results/UP000002485_284812_SCHPO_v4_and_vDulab_and_vDulab2_annotated_PAE_domain_based_on_pLDDT.tsv", sep="\t")

In [20]:
coding_genes

,gene_systematic_id,gene_systematic_id_with_prefix,gene_name,chromosome_id,gene_product,uniprot_id,gene_type,synonyms,Pombase_seq,Pombase_length,AF_seq,AF_length,type,Source,transcript_id,DY47073_seq,DY47073_length,DY47073_type
0,SPAC1002.01,PomBase:SPAC1002.01,mrx11,chromosome_1,mitochondrial expression network (MIOREX) comp...,Q9US57,protein coding gene,SPAC1610.05,MLPPTIRISGLAKTLHIPSRSPLQALKGSFILLNKRKFHYSPFILQ...,162.0,MLPPTIRISGLAKTLHIPSRSPLQALKGSFILLNKRKFHYSPFILQ...,179.0,different_seq_different_length,vDulab,SPAC1002.01.1,MLPPTIRISGLAKTLHIPSRSPLQALKGSFILLNKRKFHYSPFILQ...,162.0,same_seq
1,SPAC1002.02,PomBase:SPAC1002.02,pom34,chromosome_1,nucleoporin Pom34,Q9US56,protein coding gene,mug31,MASTFSQSVFARSLYEDSAENKVDSSKNTEANFPITLPKVLPTDPK...,229.0,MASTFSQSVFARSLYEDSAENKVDSSKNTEANFPITLPKVLPTDPK...,229.0,same_seq,AFdb,SPAC1002.02.1,MASTFSQSVFARSLYEDSAENKVDSSKNTEANFPITLPKVLPTDPK...,229.0,same_seq
2,SPAC1002.03c,PomBase:SPAC1002.03c,gls2,chromosome_1,glucosidase II alpha subunit Gls2,Q9US55,protein coding gene,NaN,MRYHGICWFIFQAAIIFAIFGSCQGAFRHQFKTAEQDGFARRNRDL...,923.0,MRYHGICWFIFQAAIIFAIFGSCQGAFRHQFKTAEQDGFARRNRDL...,923.0,same_seq,AFdb,SPAC1002.03c.1,MRYHGICWFIFQAAIIFAIFGSCQGAFRHQFKTAEQDGFARRNRDL...,923.0,same_seq
3,SPAC1002.04c,PomBase:SPAC1002.04c,taf11,chromosome_1,transcription factor TFIID complex subunit Taf11,Q9US54,protein coding gene,NaN,MKRTGSASSVPRFQRSMSYDQSNSPTAGAHAKANTASPAAEPAPSS...,199.0,MKRTGSASSVPRFQRSMSYDQSNSPTAGAHAKANTASPAAEPAPSS...,199.0,same_seq,AFdb,SPAC1002.04c.1,MKRTGSASSVPRFQRSMSYDQSNSPTAGAHAKANTASPAAEPAPSS...,199.0,same_seq
4,SPAC1002.05c,PomBase:SPAC1002.05c,jmj2,chromosome_1,histone demethylase Jmj2,Q9US53,protein coding gene,kdm5,MSKSTFHQTSPSFVQYTHATDKLKPAPHGMNTRGNNLSSMSKNTGQ...,715.0,MSKSTFHQTSPSFVQYTHATDKLKPAPHGMNTRGNNLSSMSKNTGQ...,715.0,same_seq,AFdb,SPAC1002.05c.1,MSKSTFHQTSPSFVQYTHATDKLKPAPHGMNTRGNNLSSMSKNTGQ...,715.0,same_seq
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5181,SPAC212.05c_1,NaN,SPAC212.05c_1,NaN,NaN,SPAC212.05c_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vDulab2,SPAC212.05c.1_1,IMAFGLKVEPSGGGVLTSDSLPTHNGKTALQSDATVGSDIQQIELQ...,63.0,no_this_protein_in_pombase_or_AFdb
5182,SPAC212.11_1,NaN,SPAC212.11_1,NaN,NaN,SPAC212.11_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vDulab2,SPAC212.11.1_1,MVVASEIAKVASKTARDIAGCFTCQCGTQFDNVERIVQHFKECRYR...,1927.0,no_this_protein_in_pombase_or_AFdb
5183,SPMTR.02_1,NaN,SPMTR.02_1,NaN,NaN,SPMTR.02_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vDulab2,SPMTR.02.1_1,MKRVAVLLKTVMCEFLKCDYNGYDRIISLLRRILTLICTPNLNGLT...,159.0,no_this_protein_in_pombase_or_AFdb
5184,SPAC2E1P3.04_1,NaN,SPAC2E1P3.04_1,NaN,NaN,SPAC2E1P3.04_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vDulab2,SPAC2E1P3.04.1_1,MAYYPHLPYHHHHHTSVPGERLSDPLDPLSADELKLAVEIIRHEYP...,712.0,no_this_protein_in_pombase_or_AFdb


In [21]:
annotated_PAE_domain

,Systematic ID,uniprot_id,domain_id,domain_residues,protein_length,domain_length,mean_pLDDT,fraction_gt_70,location,domain_type
0,SPBC460.01c,B5BP45,1,1-43,572.0,43.0,37.21,0.00,N-terminal,disordered
1,SPBC460.01c,B5BP45,2,44-548,572.0,505.0,88.38,0.96,middle,domain
2,SPBC460.01c,B5BP45,3,549-572,572.0,24.0,52.30,0.00,C-terminal,disordered
3,SPBC460.02c,B5BP46,1,1-220,220.0,220.0,95.96,1.00,Whole,domain
4,SPBC460.03,B5BP47,1,1-52,567.0,52.0,31.53,0.00,N-terminal,disordered
...,...,...,...,...,...,...,...,...,...,...
17888,braker_SPAC212.11,braker_SPAC212.11,9,1578-1913,1913.0,336.0,30.97,0.00,C-terminal,disordered
17889,SPBC3B8.10,ina17,1,1-28,86.0,28.0,61.55,0.00,N-terminal,disordered
17890,SPBC3B8.10,ina17,2,29-86,86.0,58.0,90.58,0.93,C-terminal,domain
17891,SPAC144.19,prl46,1,1-86,86.0,86.0,74.67,0.80,Whole,domain
